# Linear Regression: Foundation of Supervised Learning

## Overview
Linear regression is the simplest and most fundamental supervised learning algorithm. It models the relationship between input features and a continuous output variable using a linear function.

## Learning Objectives
- Understand the mathematical foundation of linear regression
- Implement linear regression from scratch using NumPy
- Use scikit-learn for production implementations
- Visualize regression results and residuals
- Evaluate model performance using appropriate metrics

## Mathematical Foundation

### The Linear Model
For a single feature:
$$y = \beta_0 + \beta_1 x + \epsilon$$

For multiple features:
$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n + \epsilon$$

In matrix form:
$$\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\epsilon}$$

### Cost Function (Mean Squared Error)
$$J(\boldsymbol{\beta}) = \frac{1}{2m}\sum_{i=1}^{m}(h_\boldsymbol{\beta}(\mathbf{x}^{(i)}) - y^{(i)})^2$$

### Normal Equation (Closed-form Solution)
$$\boldsymbol{\beta} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

### Gradient Descent Update Rule
$$\beta_j := \beta_j - \alpha \frac{\partial J}{\partial \beta_j}$$

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Simple Linear Regression (One Feature)

Let's start with the simplest case: predicting a target variable from a single feature.

In [ ]:
# Generate synthetic data
np.random.seed(42)
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X + np.random.randn(100, 1)

# Visualize the data
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, edgecolors='k')
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Simple Linear Regression Data', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Data shape: X={X.shape}, y={y.shape}")
print(f"True relationship: y = 4 + 3*X + noise")

## 2. Implementation from Scratch

### Method 1: Normal Equation (Closed-form Solution)

In [ ]:
class LinearRegressionNormal:
    """Linear Regression using Normal Equation"""
    
    def __init__(self):
        self.theta = None
    
    def fit(self, X, y):
        """
        Fit the model using the normal equation.
        
        Parameters:
        -----------
        X : array-like, shape (m, n)
            Training features
        y : array-like, shape (m, 1)
            Target values
        """
        # Add bias term (column of ones)
        m = X.shape[0]
        X_b = np.c_[np.ones((m, 1)), X]  # Add x0 = 1 to each instance
        
        # Normal equation: theta = (X^T X)^-1 X^T y
        self.theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
        
        return self
    
    def predict(self, X):
        """
        Make predictions using the fitted model.
        
        Parameters:
        -----------
        X : array-like, shape (m, n)
            Features to predict on
        
        Returns:
        --------
        y_pred : array, shape (m, 1)
            Predicted values
        """
        m = X.shape[0]
        X_b = np.c_[np.ones((m, 1)), X]
        return X_b @ self.theta

# Fit the model
model_normal = LinearRegressionNormal()
model_normal.fit(X, y)

print("Fitted parameters (Normal Equation):")
print(f"  Intercept (β₀): {model_normal.theta[0][0]:.4f}")
print(f"  Coefficient (β₁): {model_normal.theta[1][0]:.4f}")
print(f"\nTrue parameters: β₀=4, β₁=3")

### Method 2: Gradient Descent

In [ ]:
class LinearRegressionGD:
    """Linear Regression using Gradient Descent"""
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.theta = None
        self.cost_history = []
    
    def fit(self, X, y):
        """
        Fit the model using gradient descent.
        
        Parameters:
        -----------
        X : array-like, shape (m, n)
            Training features
        y : array-like, shape (m, 1)
            Target values
        """
        m, n = X.shape
        
        # Add bias term
        X_b = np.c_[np.ones((m, 1)), X]
        
        # Initialize parameters randomly
        self.theta = np.random.randn(n + 1, 1)
        
        # Gradient descent
        for iteration in range(self.n_iterations):
            # Compute predictions
            y_pred = X_b @ self.theta
            
            # Compute cost (MSE)
            cost = (1 / (2 * m)) * np.sum((y_pred - y) ** 2)
            self.cost_history.append(cost)
            
            # Compute gradients
            gradients = (1 / m) * X_b.T @ (y_pred - y)
            
            # Update parameters
            self.theta -= self.learning_rate * gradients
        
        return self
    
    def predict(self, X):
        """
        Make predictions using the fitted model.
        
        Parameters:
        -----------
        X : array-like, shape (m, n)
            Features to predict on
        
        Returns:
        --------
        y_pred : array, shape (m, 1)
            Predicted values
        """
        m = X.shape[0]
        X_b = np.c_[np.ones((m, 1)), X]
        return X_b @ self.theta

# Fit the model
model_gd = LinearRegressionGD(learning_rate=0.1, n_iterations=1000)
model_gd.fit(X, y)

print("Fitted parameters (Gradient Descent):")
print(f"  Intercept (β₀): {model_gd.theta[0][0]:.4f}")
print(f"  Coefficient (β₁): {model_gd.theta[1][0]:.4f}")

In [ ]:
# Visualize cost function convergence
plt.figure(figsize=(10, 6))
plt.plot(model_gd.cost_history)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Cost (MSE)', fontsize=12)
plt.title('Gradient Descent: Cost Function Convergence', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Initial cost: {model_gd.cost_history[0]:.4f}")
print(f"Final cost: {model_gd.cost_history[-1]:.4f}")

## 3. Visualization of Results

In [ ]:
# Make predictions
X_new = np.array([[0], [2]])
y_pred_normal = model_normal.predict(X_new)
y_pred_gd = model_gd.predict(X_new)

# Visualize results
plt.figure(figsize=(12, 5))

# Normal Equation
plt.subplot(1, 2, 1)
plt.scatter(X, y, alpha=0.6, edgecolors='k', label='Data')
plt.plot(X_new, y_pred_normal, 'r-', linewidth=2, label='Normal Equation')
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Linear Regression: Normal Equation', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# Gradient Descent
plt.subplot(1, 2, 2)
plt.scatter(X, y, alpha=0.6, edgecolors='k', label='Data')
plt.plot(X_new, y_pred_gd, 'b-', linewidth=2, label='Gradient Descent')
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Linear Regression: Gradient Descent', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Using Scikit-learn (Production Implementation)

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and fit the model
model_sklearn = LinearRegression()
model_sklearn.fit(X_train, y_train)

# Make predictions
y_train_pred = model_sklearn.predict(X_train)
y_test_pred = model_sklearn.predict(X_test)

print("Scikit-learn Linear Regression:")
print(f"  Intercept: {model_sklearn.intercept_[0]:.4f}")
print(f"  Coefficient: {model_sklearn.coef_[0][0]:.4f}")
print(f"\nTraining Set Metrics:")
print(f"  MSE: {mean_squared_error(y_train, y_train_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")
print(f"  MAE: {mean_absolute_error(y_train, y_train_pred):.4f}")
print(f"  R² Score: {r2_score(y_train, y_train_pred):.4f}")
print(f"\nTest Set Metrics:")
print(f"  MSE: {mean_squared_error(y_test, y_test_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")
print(f"  MAE: {mean_absolute_error(y_test, y_test_pred):.4f}")
print(f"  R² Score: {r2_score(y_test, y_test_pred):.4f}")

## 5. Multiple Linear Regression

Now let's work with multiple features.

In [ ]:
# Generate data with multiple features
np.random.seed(42)
m = 200
X_multi = 2 * np.random.rand(m, 3)  # 3 features
y_multi = 4 + 3 * X_multi[:, 0] + 2 * X_multi[:, 1] - X_multi[:, 2] + np.random.randn(m)
y_multi = y_multi.reshape(-1, 1)

print(f"Multiple features dataset shape: X={X_multi.shape}, y={y_multi.shape}")
print(f"True relationship: y = 4 + 3*X₁ + 2*X₂ - 1*X₃ + noise")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

# Fit model
model_multi = LinearRegression()
model_multi.fit(X_train, y_train)

# Make predictions
y_train_pred = model_multi.predict(X_train)
y_test_pred = model_multi.predict(X_test)

print("Multiple Linear Regression Results:")
print(f"  Intercept: {model_multi.intercept_[0]:.4f}")
print(f"  Coefficients: {model_multi.coef_[0]}")
print(f"  (True coefficients: [3, 2, -1])")
print(f"\nTest Set Performance:")
print(f"  R² Score: {r2_score(y_test, y_test_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")

## 6. Residual Analysis

Analyzing residuals is crucial for understanding model performance and detecting patterns.

In [ ]:
# Calculate residuals
residuals_train = y_train - y_train_pred
residuals_test = y_test - y_test_pred

# Create residual plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Predicted vs Actual
axes[0, 0].scatter(y_test, y_test_pred, alpha=0.6, edgecolors='k')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Values', fontsize=12)
axes[0, 0].set_ylabel('Predicted Values', fontsize=12)
axes[0, 0].set_title('Predicted vs Actual', fontsize=14)
axes[0, 0].grid(True, alpha=0.3)

# 2. Residuals vs Predicted
axes[0, 1].scatter(y_test_pred, residuals_test, alpha=0.6, edgecolors='k')
axes[0, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0, 1].set_xlabel('Predicted Values', fontsize=12)
axes[0, 1].set_ylabel('Residuals', fontsize=12)
axes[0, 1].set_title('Residual Plot', fontsize=14)
axes[0, 1].grid(True, alpha=0.3)

# 3. Histogram of Residuals
axes[1, 0].hist(residuals_test, bins=20, edgecolor='k', alpha=0.7)
axes[1, 0].set_xlabel('Residuals', fontsize=12)
axes[1, 0].set_ylabel('Frequency', fontsize=12)
axes[1, 0].set_title('Distribution of Residuals', fontsize=14)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Q-Q Plot
from scipy import stats
stats.probplot(residuals_test.flatten(), dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot', fontsize=14)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residual Statistics:")
print(f"  Mean: {residuals_test.mean():.6f} (should be close to 0)")
print(f"  Std Dev: {residuals_test.std():.4f}")

## 7. Real-World Example: Housing Prices

Let's apply linear regression to a real-world dataset.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler

# Load California housing dataset
housing = fetch_california_housing()
X_housing = pd.DataFrame(housing.data, columns=housing.feature_names)
y_housing = pd.DataFrame(housing.target, columns=['MedHouseVal'])

print("California Housing Dataset:")
print(f"Shape: {X_housing.shape}")
print(f"\nFeatures: {list(X_housing.columns)}")
print(f"\nFirst few rows:")
print(X_housing.head())
print(f"\nTarget variable (Median House Value in $100,000):")
print(y_housing.head())

In [ ]:
# Feature scaling (important for gradient descent)
scaler = StandardScaler()
X_housing_scaled = scaler.fit_transform(X_housing)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_housing_scaled, y_housing, test_size=0.2, random_state=42
)

# Train model
model_housing = LinearRegression()
model_housing.fit(X_train, y_train)

# Evaluate
y_train_pred = model_housing.predict(X_train)
y_test_pred = model_housing.predict(X_test)

print("Housing Price Prediction Results:")
print(f"\nTraining Set:")
print(f"  R² Score: {r2_score(y_train, y_train_pred):.4f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_train, y_train_pred)) * 100000:.2f}")
print(f"\nTest Set:")
print(f"  R² Score: {r2_score(y_test, y_test_pred):.4f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_test, y_test_pred)) * 100000:.2f}")

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': housing.feature_names,
    'Coefficient': model_housing.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print(f"\nFeature Importance (sorted by absolute coefficient):")
print(feature_importance)

In [ ]:
# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'])
plt.xlabel('Coefficient Value', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance in Housing Price Prediction', fontsize=14)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Two solution methods**: Normal equation (closed-form) and gradient descent (iterative)
2. **Normal equation** is faster for small datasets but becomes computationally expensive for large feature spaces
3. **Gradient descent** scales better but requires hyperparameter tuning (learning rate, iterations)
4. **Feature scaling** is crucial for gradient descent convergence
5. **Residual analysis** helps identify model assumptions and potential improvements
6. **R² score** measures the proportion of variance explained by the model
7. **RMSE** gives interpretable error in the same units as the target variable

## Next Steps

- **Polynomial Regression**: Handle non-linear relationships
- **Regularization**: Ridge and Lasso regression to prevent overfitting
- **Feature Engineering**: Create better features for improved predictions

## Exercises

1. Modify the gradient descent learning rate and observe its effect on convergence
2. Implement mini-batch gradient descent
3. Add polynomial features to the California housing dataset
4. Compare performance with and without feature scaling
5. Try different train/test split ratios and observe the impact on model performance